# Проверка потери строк в `clustering_agg_v4_05.ipynb`

Этот ноутбук повторяет подготовку данных из `clustering_agg_v4_05.ipynb` ровно до строки с фильтром:

```python
casco_hashtags_full = casco_hashtags_full[casco_hashtags_full['SUM'] > 0]
```

Цель: проверить, действительно ли строки пропали потому, что `SUM == 0` или `SUM <= 0` по колонкам `TAG_`.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from config import DATA_CSV_PART1, TARGET_COL

KEY_COL = 'POLICY'
CLAIMS_COL = 'CLAIMS_PART_DAM_COUNT'
TAG_JOIN_COL = 'TAG_JOIN_IND'

AUDIT_DIR = Path('artifacts/audit_v4_05')
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_CSV_PART1:', DATA_CSV_PART1)
print('AUDIT_DIR:', AUDIT_DIR)

## 1. Читаем исходный файл как в `_05`

In [ ]:
raw = pd.read_csv(DATA_CSV_PART1, encoding='cp1251', delimiter='^')
raw = raw.copy()
raw['_row_in_source_file'] = range(len(raw))

print(f'Строк в исходном DATA_CSV_PART1: {len(raw):,}')
print(f'Колонок в исходном DATA_CSV_PART1: {raw.shape[1]:,}')
display(raw.head())

In [ ]:
required_cols = [KEY_COL]
missing_required = [col for col in required_cols if col not in raw.columns]

if missing_required:
    raise KeyError(f'В исходном файле нет обязательных колонок: {missing_required}')

key_stats = pd.DataFrame({
    'metric': [
        'rows',
        f'non_null_{KEY_COL}',
        f'unique_{KEY_COL}',
        f'duplicate_{KEY_COL}_rows',
        f'missing_{KEY_COL}',
    ],
    'value': [
        len(raw),
        raw[KEY_COL].notna().sum(),
        raw[KEY_COL].nunique(dropna=True),
        len(raw) - raw[KEY_COL].nunique(dropna=True),
        raw[KEY_COL].isna().sum(),
    ],
})

display(key_stats)

## 2. Повторяем подготовку из `_05` по шагам

In [ ]:
def get_keys(df):
    if KEY_COL in df.columns:
        return df[KEY_COL]
    return df.index.to_series()


def collect_stats(df, step):
    keys = get_keys(df)
    rows = len(df)
    unique_keys = keys.nunique(dropna=True)
    non_null_keys = keys.notna().sum()
    return {
        'step': step,
        'rows': rows,
        f'non_null_{KEY_COL}': non_null_keys,
        f'unique_{KEY_COL}': unique_keys,
        f'duplicate_{KEY_COL}_rows': rows - unique_keys,
        f'missing_{KEY_COL}': rows - non_null_keys,
    }


steps = []

casco_hashtags_full = raw.copy()
steps.append(collect_stats(casco_hashtags_full, '01 read DATA_CSV_PART1'))

if CLAIMS_COL in casco_hashtags_full.columns:
    casco_hashtags_full[TARGET_COL] = casco_hashtags_full[CLAIMS_COL].astype(bool).astype(int)
    steps.append(collect_stats(casco_hashtags_full, '02 create target'))
else:
    print(f'Колонка {CLAIMS_COL} не найдена; для аудита потери строк target не нужен, шаг пропущен.')
    steps.append(collect_stats(casco_hashtags_full, '02 target skipped: claims column not found'))

casco_hashtags_full.set_index(KEY_COL, inplace=True)
steps.append(collect_stats(casco_hashtags_full, '03 set_index(POLICY_ZV)'))

if TAG_JOIN_COL in casco_hashtags_full.columns:
    casco_hashtags_full.drop(columns=[TAG_JOIN_COL], inplace=True)
steps.append(collect_stats(casco_hashtags_full, '04 drop TAG_JOIN_IND'))

tag_cols = casco_hashtags_full.filter(like='TAG_').columns.tolist()
if not tag_cols:
    raise ValueError('После drop TAG_JOIN_IND не найдено ни одной колонки TAG_.')

tag_values_raw = casco_hashtags_full[tag_cols]
tag_values_for_parse = tag_values_raw.copy()
object_tag_cols = tag_values_for_parse.select_dtypes(include='object').columns
if len(object_tag_cols) > 0:
    tag_values_for_parse[object_tag_cols] = tag_values_for_parse[object_tag_cols].replace(',', '.', regex=True)

tag_values_numeric = tag_values_for_parse.apply(pd.to_numeric, errors='coerce')
non_empty_tag_values = int(tag_values_raw.notna().sum().sum())
parsed_tag_values = int(tag_values_numeric.notna().sum().sum())
not_parsed_tag_values = non_empty_tag_values - parsed_tag_values
if not_parsed_tag_values > 0:
    print(f'Внимание: не удалось преобразовать в число {not_parsed_tag_values:,} непустых TAG-значений; для SUM они заменены на 0.')

tag_values = tag_values_numeric.fillna(0)
casco_hashtags_full['SUM'] = tag_values.sum(axis=1)
casco_hashtags_full['NONZERO_TAG_COUNT'] = tag_values.ne(0).sum(axis=1)
steps.append(collect_stats(casco_hashtags_full, '05 calculate SUM over TAG_ columns'))

before_filter = casco_hashtags_full.copy()
casco_hashtags_full = casco_hashtags_full[casco_hashtags_full['SUM'] > 0].copy()
steps.append(collect_stats(casco_hashtags_full, '06 filter SUM > 0'))

steps_df = pd.DataFrame(steps)
steps_df['rows_lost_from_previous_step'] = steps_df['rows'].shift(1).sub(steps_df['rows']).fillna(0).astype(int)
steps_df['rows_lost_from_raw'] = len(raw) - steps_df['rows']

steps_df.to_csv(AUDIT_DIR / 'row_count_by_step.csv', index=False, encoding='utf-8-sig')

print(f'Колонок TAG_, участвующих в SUM: {len(tag_cols):,}')
display(steps_df)

## 3. Проверяем, какие строки удалил фильтр `SUM > 0`

In [ ]:
sum_eq_0_mask = before_filter['SUM'].eq(0)
sum_lt_0_mask = before_filter['SUM'].lt(0)
sum_le_0_mask = before_filter['SUM'].le(0)
deleted_by_filter = before_filter.loc[sum_le_0_mask].copy()

loss_check = pd.DataFrame({
    'metric': [
        'rows_before_filter',
        'rows_after_filter_SUM_gt_0',
        'rows_deleted_by_filter',
        'rows_with_SUM_eq_0',
        'rows_with_SUM_lt_0',
        'rows_with_SUM_le_0',
        'deleted_rows_equal_SUM_le_0_rows',
        'deleted_rows_equal_SUM_eq_0_rows',
    ],
    'value': [
        len(before_filter),
        len(casco_hashtags_full),
        len(before_filter) - len(casco_hashtags_full),
        int(sum_eq_0_mask.sum()),
        int(sum_lt_0_mask.sum()),
        int(sum_le_0_mask.sum()),
        (len(before_filter) - len(casco_hashtags_full)) == int(sum_le_0_mask.sum()),
        (len(before_filter) - len(casco_hashtags_full)) == int(sum_eq_0_mask.sum()),
    ],
})

loss_check.to_csv(AUDIT_DIR / 'sum_loss_check.csv', index=False, encoding='utf-8-sig')
display(loss_check)

if int(sum_lt_0_mask.sum()) == 0:
    print('Вывод: фильтр удалил строки с SUM == 0.')
else:
    print('Вывод: фильтр удалил строки с SUM <= 0; есть строки с отрицательным SUM, их надо отдельно проверить.')

In [ ]:
sum_distribution = (
    before_filter['SUM']
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis('SUM')
    .reset_index(name='rows')
)
sum_distribution['share'] = sum_distribution['rows'] / len(before_filter)
sum_distribution.to_csv(AUDIT_DIR / 'sum_distribution.csv', index=False, encoding='utf-8-sig')

display(before_filter['SUM'].describe().to_frame('SUM'))
display(sum_distribution.head(30))

## 4. Сохраняем удаленные строки и примеры

In [ ]:
deleted_reset = deleted_by_filter.reset_index()

sample_cols = [
    KEY_COL,
    '_row_in_source_file',
    CLAIMS_COL,
    TARGET_COL,
    'SUM',
    'NONZERO_TAG_COUNT',
]
sample_cols = [col for col in sample_cols if col in deleted_reset.columns]

deleted_key_cols = [KEY_COL, '_row_in_source_file', 'SUM', 'NONZERO_TAG_COUNT']
deleted_key_cols = [col for col in deleted_key_cols if col in deleted_reset.columns]

deleted_reset[deleted_key_cols].to_csv(
    AUDIT_DIR / 'deleted_policy_zv_sum_le_0.csv',
    index=False,
    encoding='utf-8-sig',
)
deleted_reset[sample_cols].head(1000).to_csv(
    AUDIT_DIR / 'deleted_rows_sum_le_0_sample.csv',
    index=False,
    encoding='utf-8-sig',
)

print(f'Удаленных строк: {len(deleted_reset):,}')
print('Файлы сохранены:')
print(AUDIT_DIR / 'deleted_policy_zv_sum_le_0.csv')
print(AUDIT_DIR / 'deleted_rows_sum_le_0_sample.csv')
display(deleted_reset[sample_cols].head(20))

## 5. Опционально: сверяем с сохраненным итоговым CSV

Этот блок полезен, если `artifacts/csv/all_groups_agg_coef.csv` был создан именно запуском `clustering_agg_v4_05.ipynb`. Если файл был создан другим ноутбуком, например `clustering_agg_v4.ipynb`, сравнение может отличаться.

In [ ]:
OUTPUT_PATH = Path('artifacts/csv/all_groups_agg_coef.csv')

if OUTPUT_PATH.exists():
    result_csv = pd.read_csv(OUTPUT_PATH, encoding='utf-8-sig')
    unnamed_cols = [col for col in result_csv.columns if str(col).startswith('Unnamed')]
    if KEY_COL not in result_csv.columns and unnamed_cols:
        result_csv = result_csv.rename(columns={unnamed_cols[0]: KEY_COL})

    if KEY_COL not in result_csv.columns:
        print(f'В {OUTPUT_PATH} не найдена колонка {KEY_COL}; сверка пропущена.')
    else:
        expected_counts = casco_hashtags_full.reset_index().groupby(KEY_COL, dropna=False).size().rename('expected_rows')
        result_counts = result_csv.groupby(KEY_COL, dropna=False).size().rename('result_rows')

        compare_counts = pd.concat([expected_counts, result_counts], axis=1).fillna(0).astype(int)
        compare_counts['delta_result_minus_expected'] = compare_counts['result_rows'] - compare_counts['expected_rows']

        compare_summary = pd.DataFrame({
            'metric': [
                'expected_rows_after_SUM_gt_0',
                'result_csv_rows',
                'row_delta_result_minus_expected',
                'POLICY_ZV_with_missing_result_rows',
                'POLICY_ZV_with_extra_result_rows',
                'result_matches_expected_counts',
            ],
            'value': [
                len(casco_hashtags_full),
                len(result_csv),
                len(result_csv) - len(casco_hashtags_full),
                int(compare_counts['delta_result_minus_expected'].lt(0).sum()),
                int(compare_counts['delta_result_minus_expected'].gt(0).sum()),
                bool(compare_counts['delta_result_minus_expected'].eq(0).all()),
            ],
        })

        compare_summary.to_csv(AUDIT_DIR / 'result_csv_compare_summary.csv', index=False, encoding='utf-8-sig')
        compare_counts.query('delta_result_minus_expected != 0').reset_index().to_csv(
            AUDIT_DIR / 'result_csv_policy_zv_count_differences.csv',
            index=False,
            encoding='utf-8-sig',
        )

        display(compare_summary)
        display(compare_counts.query('delta_result_minus_expected != 0').head(20))
else:
    print(f'Файл {OUTPUT_PATH} не найден; сверка с итоговым CSV пропущена.')

## Что смотреть после запуска

- `row_count_by_step.csv` — на каком шаге изменилось число строк.
- `sum_loss_check.csv` — равно ли число удаленных строк числу строк с `SUM == 0` или `SUM <= 0`.
- `deleted_policy_zv_sum_le_0.csv` — все удаленные `POLICY_ZV`.
- `deleted_rows_sum_le_0_sample.csv` — пример удаленных строк.